[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lea-2005/tpi-analisis-de-datos-grupo-3/blob/main/notebooks/TPI_Analisis_de_Datos.ipynb)

# TPI — Introducción al Análisis de Datos
## Análisis de la cancelación de clientes (Churn) en una empresa de telecomunicaciones

**División:** 143 · **Docente:** Zotti, Enzo Martín · **Fecha:** octubre 2026

**Integrantes:**
- Mamani, Leandro
- Leyes, Agustín
- Guantay, Marcos
- Verdecanna, Luciano

> **Dataset:** *Telco Customer Churn* (Kaggle / IBM Sample Data Sets).
> Este notebook está pensado para ejecutarse de principio a fin en Google Colab: *Entorno de ejecución → Ejecutar todas*.

---
# Parte A — Fundamentos, Ingesta y Limpieza de Datos

## A.0 Configuración del entorno
Importamos las librerías y fijamos opciones de visualización para que las salidas sean legibles y reproducibles.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Semilla fija: cualquier operación aleatoria (muestreos, jitter de gráficos) da siempre el mismo resultado
SEMILLA = 42
np.random.seed(SEMILLA)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", palette="deep")

print(f"pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}")

## A.1 Fundamentos

### A.1.1 Problema de negocio
Una compañía de telecomunicaciones pierde cada mes una parte de su cartera de clientes, que dan de baja el servicio (*churn*). Captar un cliente nuevo cuesta varias veces más que retener uno existente, por lo que cada baja implica perder el ingreso mensual recurrente y además la inversión hecha para adquirirlo.

**Pregunta central:** *¿Qué características sociodemográficas, de contratación y de facturación distinguen a los clientes que cancelan el servicio de los que permanecen?*

Responder esta pregunta permite al área comercial **identificar segmentos de riesgo** y orientar acciones de retención (bonificaciones, migración a contratos más largos, paquetes de servicios) hacia donde tienen mayor impacto.

### A.1.2 Variable de interés
La variable objetivo del estudio es **`Churn`** (*Yes* / *No*): indica si el cliente dio de baja el servicio en el último mes. Todo el análisis se organiza para comparar la **tasa de cancelación** entre grupos de clientes.

### A.1.3 Hipótesis analíticas
Formulamos las siguientes hipótesis, que se contrastarán en el análisis exploratorio (Parte B):

| # | Hipótesis | Variables involucradas |
|---|-----------|------------------------|
| H1 | Los clientes con contrato **mes a mes** cancelan en mayor proporción que los de contrato anual o bianual, porque no tienen costo de salida. | `Contract`, `Churn` |
| H2 | La probabilidad de baja es **mayor en los primeros meses** de relación y disminuye con la antigüedad. | `tenure`, `Churn` |
| H3 | Los clientes con **cargos mensuales altos** cancelan más, por sensibilidad al precio. | `MonthlyCharges`, `Churn` |
| H4 | Los clientes que pagan con **cheque electrónico** (pago manual, no automático) presentan mayor churn. | `PaymentMethod`, `Churn` |
| H5 | Los clientes **sin servicios de soporte o seguridad** (`TechSupport`, `OnlineSecurity`) cancelan más, por menor vinculación con la empresa. | `TechSupport`, `OnlineSecurity`, `Churn` |
| H6 | Los clientes **adultos mayores** y los que **no tienen pareja ni dependientes** muestran mayor tasa de baja. | `SeniorCitizen`, `Partner`, `Dependents`, `Churn` |

## A.2 Ingesta de datos
Cargamos el CSV desde el repositorio público de IBM (misma fuente que el dataset de Kaggle), lo que permite ejecutar el notebook en Colab sin credenciales. Si existe una copia local en `data/raw/`, se usa esa.

In [ ]:
RUTA_LOCAL = "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
URL_DATOS = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
             "master/data/Telco-Customer-Churn.csv")

fuente = RUTA_LOCAL if os.path.exists(RUTA_LOCAL) else URL_DATOS
df_raw = pd.read_csv(fuente)

print(f"Fuente: {fuente}")
print(f"Dimensiones: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
df_raw.head()

In [ ]:
df_raw.info()

Cada **fila representa un cliente** y cada **columna un atributo** de ese cliente. El dataset tiene 7.043 clientes y 21 variables. A simple vista, `info()` no muestra valores nulos, pero `TotalCharges` aparece como texto (`object`) cuando debería ser numérica: es una primera señal de un problema de calidad que se trata en A.4.

## A.3 Clasificación del dataset y de las variables

### A.3.1 Tipo de datos según su estructura
El dataset **Telco Customer Churn es un conjunto de datos estructurado**, porque:
- Está organizado en **formato tabular** (archivo CSV) con filas y columnas.
- Tiene un **esquema fijo y predefinido**: todas las filas comparten las mismas 21 columnas, en el mismo orden.
- Cada columna contiene un **único tipo de dato** con un dominio acotado (por ejemplo, `Contract` solo admite tres valores).
- Puede cargarse directamente en una tabla relacional o en un `DataFrame` y consultarse con operaciones de filtrado, agrupación y agregación.

No es **semi-estructurado** (como JSON o XML, donde cada registro puede tener campos distintos o anidados) ni **no estructurado** (texto libre, imágenes, audio), ya que no requiere ningún proceso previo para extraer los campos.

### A.3.2 Relevamiento de variables
Clasificamos cada variable según la teoría estadística y la contrastamos con el `dtype` que Pandas le asigna por defecto al leer el CSV.

In [ ]:
clasificacion_teorica = {
    "customerID":       ("Cualitativa", "Nominal (identificador)", "Código único del cliente"),
    "gender":           ("Cualitativa", "Nominal dicotómica",      "Género del cliente"),
    "SeniorCitizen":    ("Cualitativa", "Nominal dicotómica",      "Adulto mayor (1) o no (0)"),
    "Partner":          ("Cualitativa", "Nominal dicotómica",      "Tiene pareja"),
    "Dependents":       ("Cualitativa", "Nominal dicotómica",      "Tiene personas a cargo"),
    "tenure":           ("Cuantitativa", "Discreta",               "Antigüedad en meses"),
    "PhoneService":     ("Cualitativa", "Nominal dicotómica",      "Tiene telefonía"),
    "MultipleLines":    ("Cualitativa", "Nominal",                 "Tiene múltiples líneas"),
    "InternetService":  ("Cualitativa", "Nominal",                 "Tipo de conexión a internet"),
    "OnlineSecurity":   ("Cualitativa", "Nominal",                 "Servicio de seguridad online"),
    "OnlineBackup":     ("Cualitativa", "Nominal",                 "Servicio de backup online"),
    "DeviceProtection": ("Cualitativa", "Nominal",                 "Protección de dispositivos"),
    "TechSupport":      ("Cualitativa", "Nominal",                 "Soporte técnico"),
    "StreamingTV":      ("Cualitativa", "Nominal",                 "Streaming de TV"),
    "StreamingMovies":  ("Cualitativa", "Nominal",                 "Streaming de películas"),
    "Contract":         ("Cualitativa", "Ordinal",                 "Duración del contrato"),
    "PaperlessBilling": ("Cualitativa", "Nominal dicotómica",      "Factura electrónica"),
    "PaymentMethod":    ("Cualitativa", "Nominal",                 "Medio de pago"),
    "MonthlyCharges":   ("Cuantitativa", "Continua",               "Cargo mensual (USD)"),
    "TotalCharges":     ("Cuantitativa", "Continua",               "Cargo total acumulado (USD)"),
    "Churn":            ("Cualitativa", "Nominal dicotómica",      "Canceló el servicio (variable objetivo)"),
}

tabla_variables = pd.DataFrame.from_dict(
    clasificacion_teorica, orient="index",
    columns=["tipo_teorico", "subtipo", "descripcion"])
tabla_variables["dtype_pandas"] = df_raw.dtypes.astype(str)
tabla_variables["valores_unicos"] = df_raw.nunique()

# Hay coherencia cuando una variable cuantitativa tiene dtype numérico
# y una cualitativa tiene dtype de texto.
es_numerico = df_raw.dtypes.apply(pd.api.types.is_numeric_dtype)
tabla_variables["coherente"] = np.where(
    (tabla_variables["tipo_teorico"] == "Cuantitativa") == es_numerico, "Sí", "NO")

tabla_variables

In [ ]:
print("Variables cuya clasificación teórica no coincide con el dtype de Pandas:")
tabla_variables.loc[tabla_variables["coherente"] == "NO",
                    ["tipo_teorico", "subtipo", "dtype_pandas"]]

**Resumen del relevamiento:** 3 variables cuantitativas (1 discreta y 2 continuas) y 18 cualitativas (17 nominales, entre ellas el identificador, y 1 ordinal).

**Contraste teoría vs. Pandas.** Pandas infiere el tipo a partir del contenido del archivo y no del significado de la variable. Eso produce dos discrepancias:

1. **`SeniorCitizen` → `int64`**: está codificada como 0/1, por eso Pandas la lee como número entero. Conceptualmente es una **categórica nominal dicotómica**: calcular su media o su desvío no tiene sentido de negocio (aunque la media coincide con la proporción de adultos mayores). Se convierte a categoría.
2. **`TotalCharges` → `object` (texto)**: es una variable **cuantitativa continua**, pero contiene cadenas vacías (`" "`) que impiden la conversión automática a número. Se corrige en A.4.

Otras observaciones:
- **`tenure`** se mide en meses enteros, por lo que es **discreta**, aunque se puede tratar como continua en los gráficos dado su rango (0 a 72).
- **`MonthlyCharges`** y **`TotalCharges`** son importes monetarios. En sentido estricto se miden en centavos, pero se tratan como **continuas** por convención.
- **`Contract`** es **ordinal** porque sus categorías tienen un orden natural de compromiso: *Month-to-month < One year < Two year*. Se le asignará un tipo categórico ordenado.
- Las demás variables cualitativas quedan como `object` (texto). Convertirlas a `category` reduce el uso de memoria y explicita su naturaleza.

## A.4 Limpieza y calidad de datos
Trabajamos sobre una copia (`df`) para conservar intacto el dataset original (`df_raw`) como referencia.

> **Orden de las etapas:** la limpieza se realiza **antes** de la manipulación porque las columnas calculadas y las agregaciones requieren que `TotalCharges` ya sea numérica.

### A.4.1 Valores faltantes y cadenas anómalas

In [ ]:
df = df_raw.copy()

print("Valores nulos (NaN) por columna:")
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().sum().sum() else "  -> Ninguno")

# Búsqueda de cadenas vacías o compuestas solo por espacios en columnas de texto
cols_texto = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
cadenas_vacias = {c: int(df[c].astype(str).str.strip().eq("").sum()) for c in cols_texto}
cadenas_vacias = {c: n for c, n in cadenas_vacias.items() if n > 0}
print("\nCadenas vacías / solo espacios por columna:", cadenas_vacias)

In [ ]:
filas_anomalas = df[df["TotalCharges"].astype(str).str.strip() == ""]
print(f"Registros con TotalCharges vacío: {len(filas_anomalas)} "
      f"({len(filas_anomalas) / len(df):.2%} del total)")
filas_anomalas[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

In [ ]:
print("Antigüedad (tenure) de los registros anómalos:", filas_anomalas["tenure"].unique())
print("Clientes con tenure = 0 en todo el dataset:", (df["tenure"] == 0).sum())

**Diagnóstico.** `isna()` no detecta faltantes porque el dato no está vacío técnicamente: es una cadena con un espacio (`" "`). Hay **11 registros** en esta situación (0,16 % del total), y **todos tienen `tenure = 0`**, que además son los únicos clientes con antigüedad cero. No se trata de un error aleatorio: son **clientes recién incorporados que todavía no recibieron su primera factura**, por eso su cargo acumulado no está informado.

**Decisión: imputar `TotalCharges = 0`.** Elegimos esta opción en lugar de eliminar las filas o imputar la media/mediana porque:
- Es el **valor lógicamente correcto**: si el cliente aún no fue facturado, su cargo acumulado es cero.
- Imputar la media (~USD 2.280) inventaría un historial de facturación que no existe y distorsionaría la relación `TotalCharges ≈ tenure × MonthlyCharges`.
- Eliminar las filas también sería defendible por su bajo peso (0,16 %), pero perderíamos información sobre clientes nuevos, que son relevantes para la hipótesis H2.

Primero reemplazamos las cadenas vacías por `NaN` (para que el faltante sea explícito) y después imputamos.

In [ ]:
df["TotalCharges"] = df["TotalCharges"].replace(r"^\s*$", np.nan, regex=True)
print("NaN en TotalCharges tras normalizar:", df["TotalCharges"].isna().sum())

# Limpieza preventiva: espacios sobrantes al inicio o al final en todas las columnas de texto
for c in cols_texto:
    df[c] = df[c].astype("string").str.strip()

### A.4.2 Adecuación de tipos de datos (dtypes)

In [ ]:
# 1) TotalCharges: texto -> numérico continuo, e imputación justificada en A.4.1
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").astype("float64")
df.loc[df["TotalCharges"].isna() & (df["tenure"] == 0), "TotalCharges"] = 0.0

# 2) SeniorCitizen: 0/1 -> categórica con etiquetas consistentes con el resto (Yes/No)
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

# 3) Contract: categórica ORDINAL (orden de compromiso)
df["Contract"] = pd.Categorical(
    df["Contract"], categories=["Month-to-month", "One year", "Two year"], ordered=True)

# 4) Resto de variables cualitativas -> category (excepto el identificador)
cualitativas = tabla_variables.index[
    (tabla_variables["tipo_teorico"] == "Cualitativa")
    & (tabla_variables.index != "customerID")
    & (tabla_variables.index != "Contract")]
for c in cualitativas:
    df[c] = df[c].astype("category")

# 5) Variable objetivo también en formato numérico binario, útil para calcular tasas (media = tasa)
df["Churn_flag"] = (df["Churn"] == "Yes").astype("int8")

print("Nulos restantes en todo el dataset:", int(df.isna().sum().sum()))
pd.DataFrame({"dtype_original": df_raw.dtypes.astype(str),
              "dtype_corregido": df.drop(columns="Churn_flag").dtypes.astype(str)})

In [ ]:
uso_antes = df_raw.memory_usage(deep=True).sum() / 1024**2
uso_despues = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memoria: {uso_antes:.2f} MB -> {uso_despues:.2f} MB "
      f"({1 - uso_despues / uso_antes:.0%} menos)")

Después de la conversión, los tipos de Pandas coinciden con la clasificación teórica: `tenure` es entera (discreta), `MonthlyCharges` y `TotalCharges` son `float64` (continuas), `Contract` es categórica ordenada y el resto de las cualitativas son `category`. Además se agrega `Churn_flag` (1 = canceló, 0 = no canceló), cuyo promedio en cualquier grupo es directamente la **tasa de churn**.

### A.4.3 Duplicados y consistencia de categorías

In [ ]:
print("Filas completamente duplicadas:", df.duplicated().sum())
print("customerID duplicados:", df["customerID"].duplicated().sum())

print("\nCategorías por variable cualitativa:")
for c in list(cualitativas) + ["Contract"]:
    print(f"  {c:<17} {list(df[c].cat.categories)}")

In [ ]:
# Coherencia lógica entre variables relacionadas
sin_internet = df["InternetService"] == "No"
servicios_internet = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                      "TechSupport", "StreamingTV", "StreamingMovies"]
incoherentes_int = (df.loc[sin_internet, servicios_internet] != "No internet service").any(axis=1).sum()
incoherentes_tel = (df.loc[df["PhoneService"] == "No", "MultipleLines"] != "No phone service").sum()
print("Clientes sin internet con servicios de internet informados:", incoherentes_int)
print("Clientes sin telefonía con MultipleLines informado:", incoherentes_tel)

# TotalCharges debería aproximarse a tenure x MonthlyCharges
ratio = df.loc[df["tenure"] > 0, "TotalCharges"] / (df["tenure"] * df["MonthlyCharges"])
print(f"\nRatio TotalCharges / (tenure x MonthlyCharges): mediana {ratio.median():.3f}, "
      f"rango [{ratio.min():.2f}, {ratio.max():.2f}]")

No hay registros duplicados y cada `customerID` es único. Las categorías no presentan variantes de escritura (mayúsculas, espacios, sinónimos).

Las categorías **"No internet service"** y **"No phone service"** **no son valores faltantes**: indican que el cliente no contrató el servicio base y por eso no puede tener los servicios adicionales. Lo verificamos con el control de coherencia (0 inconsistencias) y se conservan tal cual, porque aportan información.

El cociente `TotalCharges / (tenure × MonthlyCharges)` se mantiene cerca de 1, lo que confirma que el cargo total es consistente con la antigüedad y el abono mensual. Las diferencias se explican por cambios de plan o de precio a lo largo del tiempo.

### A.4.4 Gestión de valores atípicos (outliers) — método IQR
Para cada variable cuantitativa calculamos los cuartiles y los límites de Tukey:

$$\text{Lím. inf} = Q_1 - 1{,}5 \cdot IQR \qquad \text{Lím. sup} = Q_3 + 1{,}5 \cdot IQR \qquad IQR = Q_3 - Q_1$$

Los valores fuera de ese rango se consideran atípicos.

In [ ]:
numericas = ["tenure", "MonthlyCharges", "TotalCharges"]

def resumen_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((serie < li) | (serie > ls)).sum()
    return pd.Series({"Q1": q1, "Q3": q3, "IQR": iqr, "lim_inf": li, "lim_sup": ls,
                      "min": serie.min(), "max": serie.max(),
                      "n_outliers": int(n_out), "%_outliers": n_out / len(serie) * 100})

tabla_iqr = df[numericas].apply(resumen_iqr).T
tabla_iqr

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
titulos = {"tenure": "Antigüedad (meses)", "MonthlyCharges": "Cargo mensual (USD)",
           "TotalCharges": "Cargo total (USD)"}
for ax, col in zip(axes, numericas):
    sns.boxplot(x=df[col], ax=ax, color="#4C72B0", width=0.5)
    ax.axvline(tabla_iqr.loc[col, "lim_sup"], ls="--", color="crimson", lw=1, label="Límite sup. IQR")
    if tabla_iqr.loc[col, "lim_inf"] > df[col].min() - 1:
        ax.axvline(tabla_iqr.loc[col, "lim_inf"], ls="--", color="crimson", lw=1)
    ax.set_title(titulos[col])
    ax.set_xlabel("")
axes[0].legend(loc="upper right", fontsize=8)
fig.suptitle("Diagramas de caja de las variables cuantitativas", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Análisis complementario: outliers por grupo de Churn (un valor puede ser típico
# a nivel global pero atípico dentro de su grupo)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col in zip(axes, numericas):
    sns.boxplot(data=df, x="Churn", y=col, hue="Churn", ax=ax, legend=False,
                palette={"No": "#4C72B0", "Yes": "#DD8452"})
    ax.set_title(f"{titulos[col]} según Churn")
plt.tight_layout()
plt.show()

outliers_grupo = (df.groupby("Churn", observed=True)[numericas]
                    .agg(lambda s: int(resumen_iqr(s)["n_outliers"])))
print("Outliers IQR dentro de cada grupo de Churn:")
outliers_grupo

**Interpretación y decisión.**
- **A nivel global, el método IQR no detecta valores atípicos** en ninguna de las tres variables: todos los valores están dentro de los límites de Tukey. Los rangos son plausibles para el negocio: antigüedad de 0 a 72 meses (6 años) y cargo mensual entre ~USD 18 y ~USD 119, que corresponde a clientes con un solo servicio básico y con el paquete completo.
- **`TotalCharges` es fuertemente asimétrica a la derecha** (mediana muy inferior a la media). Esto no se debe a errores de carga: es el resultado de multiplicar antigüedad por abono, y muchos clientes son recientes.
- **Dentro del grupo `Churn = Yes`** aparecen algunos valores de `tenure` y `TotalCharges` por encima del bigote superior. Son **clientes antiguos y de alta facturación que igualmente se fueron**. Son casos poco frecuentes pero reales y muy relevantes para el negocio (las bajas más costosas).

**Decisión: conservar todos los valores.** No hay evidencia de errores de medición o de carga, y los extremos representan comportamientos legítimos de los clientes. Eliminarlos o recortarlos sesgaría el análisis y escondería justamente a los clientes de mayor valor perdidos. La asimetría de `TotalCharges` se tendrá en cuenta en la Parte B: se priorizará la **mediana** sobre la media como medida de tendencia central.

## A.5 Manipulación de datos con Pandas

### A.5.1 Transformación básica — renombrado de columnas
Estandarizamos los nombres a `snake_case` en minúsculas para que sean consistentes y fáciles de escribir en el código.

In [ ]:
renombres = {
    "customerID": "customer_id", "gender": "gender", "SeniorCitizen": "senior_citizen",
    "Partner": "partner", "Dependents": "dependents", "tenure": "tenure",
    "PhoneService": "phone_service", "MultipleLines": "multiple_lines",
    "InternetService": "internet_service", "OnlineSecurity": "online_security",
    "OnlineBackup": "online_backup", "DeviceProtection": "device_protection",
    "TechSupport": "tech_support", "StreamingTV": "streaming_tv",
    "StreamingMovies": "streaming_movies", "Contract": "contract",
    "PaperlessBilling": "paperless_billing", "PaymentMethod": "payment_method",
    "MonthlyCharges": "monthly_charges", "TotalCharges": "total_charges",
    "Churn": "churn", "Churn_flag": "churn_flag",
}
df = df.rename(columns=renombres)
df.columns.tolist()

### A.5.2 Transformación básica — eliminación de atributos irrelevantes
`customer_id` es un **identificador**: es único para cada fila y no tiene capacidad explicativa sobre la cancelación. Ya verificamos que no hay duplicados (A.4.3), así que lo eliminamos del conjunto de análisis.

Las demás variables se conservan porque cada una está vinculada al menos a una de las hipótesis H1 a H6. `gender` se mantiene para comprobar empíricamente si influye o no en el churn, en lugar de descartarla por supuesto.

In [ ]:
df = df.drop(columns=["customer_id"])
print(f"Dimensiones tras eliminar el identificador: {df.shape}")

### A.5.3 Transformación básica — columnas calculadas
Creamos variables derivadas mediante **operaciones aritméticas directas entre columnas**:

| Nueva columna | Fórmula | Utilidad para el negocio |
|---|---|---|
| `tenure_anios` | `tenure / 12` | Antigüedad expresada en años, más fácil de leer |
| `cargo_anual_estimado` | `monthly_charges × 12` | Ingreso anual que aporta el cliente (valor en riesgo) |
| `cargo_promedio_historico` | `total_charges / tenure` | Lo que el cliente pagó en promedio por mes a lo largo de su historia |
| `variacion_cargo` | `monthly_charges − cargo_promedio_historico` | Si es > 0, el cliente **hoy paga más** que su promedio histórico (aumento de precio o upgrade) |
| `cantidad_servicios` | suma de servicios contratados | Grado de vinculación del cliente con la empresa (H5) |

In [ ]:
df["tenure_anios"] = df["tenure"] / 12
df["cargo_anual_estimado"] = df["monthly_charges"] * 12

# Para tenure = 0 no hay historia: se usa el cargo mensual actual (evita división por cero)
df["cargo_promedio_historico"] = np.where(
    df["tenure"] > 0, df["total_charges"] / df["tenure"].replace(0, np.nan), df["monthly_charges"])
df["variacion_cargo"] = df["monthly_charges"] - df["cargo_promedio_historico"]

servicios = ["phone_service", "multiple_lines", "online_security", "online_backup",
             "device_protection", "tech_support", "streaming_tv", "streaming_movies"]
df["cantidad_servicios"] = (
    (df[servicios] == "Yes").sum(axis=1) + (df["internet_service"] != "No").astype(int))

df[["tenure", "tenure_anios", "monthly_charges", "cargo_anual_estimado", "total_charges",
    "cargo_promedio_historico", "variacion_cargo", "cantidad_servicios", "churn"]].head(8)

In [ ]:
df[["tenure_anios", "cargo_anual_estimado", "variacion_cargo", "cantidad_servicios"]].describe()

### A.5.4 Carga y selección — filtrado con múltiples condiciones lógicas
Usamos los operadores `&` (Y), `|` (O) y `~` (NO) para aislar segmentos de interés. Cada condición va entre paréntesis por la precedencia de operadores en Pandas.

In [ ]:
tasa_global = df["churn_flag"].mean()
print(f"Tasa de churn global: {tasa_global:.2%}  ({df['churn_flag'].sum():,} bajas sobre {len(df):,} clientes)\n")

def describir_segmento(nombre, mascara):
    seg = df[mascara]
    print(f"{nombre}\n  clientes: {len(seg):,} ({len(seg)/len(df):.1%} de la cartera) | "
          f"tasa de churn: {seg['churn_flag'].mean():.2%} | "
          f"cargo mensual medio: USD {seg['monthly_charges'].mean():,.2f}\n")
    return seg

# Filtro 1 (&): clientes nuevos (primer año) con contrato mes a mes  -> H1 + H2
seg_nuevos_mensual = describir_segmento(
    "F1 · Contrato mes a mes Y antigüedad <= 12 meses",
    (df["contract"] == "Month-to-month") & (df["tenure"] <= 12))

# Filtro 2 (& y |): fibra óptica Y (pago con cheque electrónico O cargo mensual > percentil 75) -> H3 + H4
p75 = df["monthly_charges"].quantile(0.75)
seg_fibra_riesgo = describir_segmento(
    f"F2 · Fibra óptica Y (cheque electrónico O cargo mensual > P75 = USD {p75:.2f})",
    (df["internet_service"] == "Fiber optic")
    & ((df["payment_method"] == "Electronic check") | (df["monthly_charges"] > p75)))

# Filtro 3 (& y |): con internet pero SIN soporte técnico NI seguridad online -> H5
seg_sin_soporte = describir_segmento(
    "F3 · Con internet Y sin soporte técnico Y sin seguridad online",
    (df["internet_service"] != "No") & (df["tech_support"] == "No") & (df["online_security"] == "No"))

# Filtro 4 (& y ~): adulto mayor, sin pareja y sin dependientes -> H6
seg_senior_solo = describir_segmento(
    "F4 · Adulto mayor Y sin pareja Y sin dependientes",
    (df["senior_citizen"] == "Yes") & ~(df["partner"] == "Yes") & (df["dependents"] == "No"))

# Filtro 5 (contraste): clientes de bajo riesgo esperado
describir_segmento(
    "F5 · Contrato de 2 años O (antigüedad > 48 meses Y pago automático)",
    (df["contract"] == "Two year")
    | ((df["tenure"] > 48) & df["payment_method"].astype(str).str.contains("automatic")));

In [ ]:
# Vista de un segmento filtrado con selección de columnas (loc con máscara + lista de columnas)
df.loc[(df["contract"] == "Month-to-month") & (df["tenure"] <= 12) & (df["churn"] == "Yes"),
       ["tenure", "contract", "internet_service", "payment_method", "monthly_charges", "churn"]].head(10)

### A.5.5 Agregación — segmentación con `groupby` y múltiples agregaciones
Usamos *named aggregations* para combinar en una sola tabla **conteos, promedios y sumas** por segmento.

In [ ]:
resumen_contrato = df.groupby("contract", observed=True).agg(
    clientes=("churn_flag", "size"),
    bajas=("churn_flag", "sum"),
    tasa_churn=("churn_flag", "mean"),
    antiguedad_media=("tenure", "mean"),
    cargo_mensual_medio=("monthly_charges", "mean"),
    facturacion_total=("total_charges", "sum"),
)
resumen_contrato["%_cartera"] = resumen_contrato["clientes"] / len(df) * 100
resumen_contrato["tasa_churn"] *= 100
resumen_contrato

In [ ]:
resumen_pago = (df.groupby("payment_method", observed=True)
                  .agg(clientes=("churn_flag", "size"),
                       tasa_churn=("churn_flag", "mean"),
                       cargo_mensual_medio=("monthly_charges", "mean"),
                       ingreso_mensual_perdido=("monthly_charges",
                                                lambda s: s[df.loc[s.index, "churn_flag"] == 1].sum()))
                  .sort_values("tasa_churn", ascending=False))
resumen_pago["tasa_churn"] *= 100
resumen_pago

In [ ]:
# Segmentación cruzada: contrato x tipo de internet
resumen_cruzado = df.groupby(["contract", "internet_service"], observed=True).agg(
    clientes=("churn_flag", "count"),
    tasa_churn=("churn_flag", "mean"),
    cargo_mensual_medio=("monthly_charges", "mean"),
    servicios_medios=("cantidad_servicios", "mean"),
)
resumen_cruzado["tasa_churn"] *= 100
resumen_cruzado.round(2)

In [ ]:
# Perfil comparativo: ¿en qué se diferencian, en promedio, los que se van de los que se quedan?
df.groupby("churn", observed=True).agg(
    clientes=("tenure", "size"),
    tenure_medio=("tenure", "mean"),
    tenure_mediana=("tenure", "median"),
    cargo_mensual_medio=("monthly_charges", "mean"),
    cargo_total_mediana=("total_charges", "median"),
    servicios_medios=("cantidad_servicios", "mean"),
    variacion_cargo_media=("variacion_cargo", "mean"),
    ingreso_mensual_total=("monthly_charges", "sum"),
)

**Primeras lecturas (se profundizan en la Parte B):**
- La **tasa de churn global es del ~26,5 %**: aproximadamente 1 de cada 4 clientes se dio de baja.
- El **contrato mes a mes** concentra más de la mitad de la cartera y tiene una tasa de cancelación muy superior a la de los contratos de 1 y 2 años, lo que apoya **H1**. El segmento *mes a mes + primer año* (F1) tiene una tasa cercana a la mitad de sus clientes, lo que también apoya **H2**.
- El **cheque electrónico** es el medio de pago con mayor churn y el que acumula más ingreso mensual perdido (**H4**).
- Los clientes que cancelan tienen, en promedio, **menor antigüedad y mayor cargo mensual** que los que permanecen (**H2**, **H3**).
- Los clientes con internet pero **sin soporte técnico ni seguridad** (F3) superan con claridad la tasa global (**H5**).

Estas diferencias son descriptivas. Su análisis estadístico y gráfico se desarrolla en la sección B.

## A.6 Dataset final de la etapa A

In [ ]:
df.info()

In [ ]:
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/telco_churn_limpio.csv", index=False)
print("Dataset limpio guardado en data/processed/telco_churn_limpio.csv")

### Resumen de decisiones de la Parte A

| Etapa | Problema / tarea | Decisión |
|---|---|---|
| Estructura | Tipo de dataset | **Estructurado** (tabular, esquema fijo, CSV) |
| Tipos | `SeniorCitizen` leído como `int64` | Convertido a categórica (*Yes/No*) |
| Tipos | `TotalCharges` leído como `object` | Convertido a `float64` |
| Tipos | `Contract` con orden natural | Categórica **ordinal** |
| Faltantes | 11 `" "` en `TotalCharges` (todos con `tenure = 0`) | Imputados con **0** (clientes sin facturación todavía) |
| Calidad | Duplicados / categorías inconsistentes | No se encontraron |
| Outliers | Método IQR en 3 variables numéricas | No hay outliers globales; los extremos dentro de *Churn = Yes* son reales → **se conservan** |
| Transformación | Nombres heterogéneos | Renombrado a `snake_case` |
| Transformación | `customerID` sin valor analítico | **Eliminado** |
| Transformación | Columnas derivadas | `tenure_anios`, `cargo_anual_estimado`, `cargo_promedio_historico`, `variacion_cargo`, `cantidad_servicios`, `churn_flag` |

El `DataFrame` **`df`** queda listo para el análisis exploratorio y la visualización (Parte B).

---

# Parte B — Análisis Exploratorio, Visualización e Integración

En esta parte se responde la pregunta de negocio de A.1.1 y se contrastan las hipótesis H1–H6 de A.1.3. Se trabaja sobre el `DataFrame` **`df`** ya limpio y renombrado de la Parte A, sin volver a cargar ni limpiar los datos: así todo el notebook usa una única versión consistente del dataset.

| Sección | Contenido |
|---|---|
| B.1 | Medidas de tendencia central, dispersión y forma de las variables numéricas |
| B.2 | Correlación lineal de Pearson entre `tenure`, `monthly_charges` y `total_charges` |
| B.3 | Frecuencias absolutas/relativas y tablas de contingencia frente a `churn` |
| B.4 | Histogramas/KDE, heatmap y barras comparativas, con interpretación de negocio |
| B.5 | Contraste de hipótesis y conclusiones |

## B.1 Análisis cuantitativo: tendencia central y dispersión

Analizamos las tres variables cuantitativas del estudio. Para cada una calculamos:

- **Tendencia central:** media y mediana.
- **Dispersión:** desvío estándar, cuartiles Q1/Q3, rango intercuartílico (RIC) y coeficiente de variación (CV = desvío / media).
- **Forma:** comparación media vs. mediana para detectar asimetría.
- **Atípicos:** conteo por el criterio de Tukey (fuera de Q1 − 1,5·RIC y Q3 + 1,5·RIC).

In [ ]:
# --- B.1 Análisis cuantitativo ---
# df ya viene limpio y renombrado de la Parte A.
variables_num = ["tenure", "monthly_charges", "total_charges"]

print("Estadísticas descriptivas de las variables numéricas:")
print(df[variables_num].describe().round(2).to_string())


# Calcula e interpreta las medidas descriptivas de una variable numérica.
def describir_numerica(nombre, serie):
    media = serie.mean()
    mediana = serie.median()
    desvio = serie.std()
    q1, q3 = serie.quantile([0.25, 0.75])
    ric = q3 - q1
    cv = desvio / media
    atipicos = int(((serie < q1 - 1.5 * ric) | (serie > q3 + 1.5 * ric)).sum())

    if media > mediana:
        asimetria = "asimetría positiva (media > mediana)"
    elif media < mediana:
        asimetria = "asimetría negativa (media < mediana)"
    else:
        asimetria = "distribución simétrica"

    if cv < 0.30:
        variabilidad = "baja"
    elif cv < 0.60:
        variabilidad = "moderada"
    else:
        variabilidad = "alta"

    texto = (
        f"{nombre}: media = {media:,.2f}, mediana = {mediana:,.2f}, "
        f"desvío estándar = {desvio:,.2f}, CV = {cv:.2f} (variabilidad {variabilidad}), "
        f"RIC = {ric:,.2f}, atípicos = {atipicos}. Presenta {asimetria}."
    )
    print(texto)
    return texto


textos_numericas = [describir_numerica(col, df[col]) for col in variables_num]

**Lectura de las medidas.**

| Variable | Media | Mediana | Desvío | CV | RIC | Atípicos IQR |
|---|---:|---:|---:|---:|---:|---:|
| `tenure` | 32,37 | 29,00 | 24,56 | 0,76 (alta) | 46,00 | 0 |
| `monthly_charges` | 64,76 | 70,35 | 30,09 | 0,46 (moderada) | 54,35 | 0 |
| `total_charges` | 2.279,73 | 1.394,55 | 2.266,79 | 0,99 (alta) | 3.388,05 | 0 |

- **`tenure`** tiene una dispersión relativa alta: conviven muchos clientes nuevos con una cola de clientes de hasta 72 meses. La media (32,37) es levemente mayor que la mediana (29), señal de una asimetría positiva moderada.
- **`monthly_charges`** es la variable más estable en términos relativos (CV 0,46) y presenta **asimetría negativa**: la mediana (70,35 USD) supera a la media (64,76 USD), porque la mayoría se concentra en planes de precio medio-alto y hay una cola de clientes con cargos bajos.
- **`total_charges`** es la más dispersa (CV 0,99) y fuertemente asimétrica a la derecha: la media (2.279,73 USD) casi duplica la mediana (1.394,55 USD). Es esperable, porque acumula antigüedad × cargo mensual. Por eso, para describirla conviene usar la **mediana**, no la media.
- **No hay atípicos por IQR en ninguna variable**, lo que coincide con la decisión de conservarlos tomada en A.4.4.

## B.2 Correlación lineal de Pearson

El coeficiente de Pearson (`r`) mide la **fuerza y el sentido de la relación lineal** entre dos variables numéricas. Varía entre −1 y 1:

- **Signo:** positivo indica que las variables crecen juntas; negativo, que se mueven en sentidos opuestos.
- **Magnitud:** |r| < 0,30 débil; 0,30 ≤ |r| < 0,70 moderada; |r| ≥ 0,70 fuerte.
- **Advertencia:** correlación no implica causalidad.


In [ ]:
# --- B.2 Correlación de Pearson ---
def magnitud_correlacion(valor):
    valor_abs = abs(valor)
    if valor_abs < 0.30:
        return "débil"
    if valor_abs < 0.70:
        return "moderada"
    return "fuerte"


CONTEXTO_CORRELACION = {
    frozenset(("tenure", "total_charges")): (
        "A mayor antigüedad, mayor gasto acumulado, porque el cargo total suma los pagos "
        "de todos los meses. Es la relación más fuerte y confirma que los clientes de largo "
        "plazo son el activo más valioso; la retención temprana protege ese valor futuro."
    ),
    frozenset(("monthly_charges", "total_charges")): (
        "Un cargo mensual más alto se asocia con un mayor gasto acumulado, aunque la relación "
        "es moderada porque el total también depende de la antigüedad. Los clientes de alto "
        "valor mensual son atractivos, pero deben gestionarse con cuidado por su mayor "
        "sensibilidad al precio."
    ),
    frozenset(("tenure", "monthly_charges")): (
        "La relación es débil: el cargo mensual explica solo una parte pequeña de la "
        "antigüedad. La permanencia depende más del tipo de contrato, los servicios "
        "contratados y la experiencia del cliente que del precio mensual aislado."
    ),
}


def describir_correlacion(var_a, var_b, valor):
    sentido = "positiva" if valor > 0 else "negativa"
    movimiento = (
        "tienden a aumentar o disminuir juntas"
        if valor > 0
        else "se mueven en sentidos opuestos"
    )
    return (
        f"{var_a} y {var_b}: r = {valor:.3f}. Correlación {sentido} de magnitud "
        f"{magnitud_correlacion(valor)}; ambas variables {movimiento}."
    )


matriz_corr = df[variables_num].corr(method="pearson")
print("Matriz de correlación de Pearson:")
print(matriz_corr.round(3).to_string())

textos_correlacion = []
for i, var_a in enumerate(variables_num):
    for var_b in variables_num[i + 1:]:
        valor = matriz_corr.loc[var_a, var_b]
        texto = describir_correlacion(var_a, var_b, valor)
        contexto = CONTEXTO_CORRELACION[frozenset((var_a, var_b))]
        print("\n" + texto)
        print("Lectura de negocio:", contexto)
        textos_correlacion.append((texto, contexto))

**Interpretación de la matriz.**

| Par de variables | r de Pearson | Magnitud | Lectura |
|---|---:|---|---|
| `tenure` – `total_charges` | **0,826** | Fuerte positiva | La antigüedad explica gran parte del gasto acumulado: cada mes adicional suma facturación. Los clientes antiguos son el activo de mayor valor y la retención temprana protege ese ingreso futuro. |
| `monthly_charges` – `total_charges` | **0,651** | Moderada positiva | Un abono mensual más alto eleva el gasto acumulado, pero el total también depende de cuántos meses permaneció el cliente. |
| `tenure` – `monthly_charges` | **0,248** | Débil positiva | El precio mensual no explica la permanencia: el churn depende más del contrato, los servicios y la experiencia que del cargo mensual aislado. |

La relación más fuerte es `tenure`–`total_charges` (r = 0,826): **el tiempo de permanencia es el principal motor del valor acumulado**. Esto refuerza la prioridad de negocio de retener desde el inicio, porque cada baja temprana no solo pierde el abono mensual, sino toda la facturación futura asociada.


## B.3 Análisis categórico

Para las variables sociodemográficas y de contratación calculamos:

1. **Frecuencias absolutas y relativas** de cada categoría.
2. **Tablas de contingencia** contra `churn`, en valores absolutos y en porcentaje dentro de cada categoría (normalización por fila). El porcentaje por fila responde directamente: *¿qué proporción de clientes de esta categoría se dio de baja?*

In [ ]:
# --- B.3 Análisis categórico ---
print("Frecuencia absoluta de churn:")
print(df["churn"].value_counts().to_string())

print("\nFrecuencia relativa de churn (%):")
print((df["churn"].value_counts(normalize=True) * 100).round(2).to_string())

tasa_global = df["churn_flag"].mean()
print(f"\nTasa global de cancelación: {tasa_global:.2%} "
      f"({int(df['churn_flag'].sum()):,} bajas sobre {len(df):,} clientes)")

# Variables solicitadas: sociodemográficas y de contratación.
variables_cat = ["gender", "senior_citizen", "partner", "dependents",
                 "contract", "payment_method"]

tablas_frecuencia = {}
tablas_contingencia_abs = {}
tablas_contingencia_rel = {}

for var in variables_cat:
    absoluta = df[var].value_counts().rename("Frecuencia absoluta")
    relativa = (df[var].value_counts(normalize=True) * 100).round(2).rename("Frecuencia relativa (%)")
    tabla = pd.concat([absoluta, relativa], axis=1)
    tablas_frecuencia[var] = tabla

    contingencia_abs = pd.crosstab(df[var], df["churn"], margins=True)
    contingencia_rel = (pd.crosstab(df[var], df["churn"], normalize="index") * 100).round(2)

    tablas_contingencia_abs[var] = contingencia_abs
    tablas_contingencia_rel[var] = contingencia_rel

    print(f"\n--- {var}: frecuencias absolutas y relativas ---")
    print(tabla.to_string())
    print(f"--- {var} vs churn: frecuencias absolutas ---")
    print(contingencia_abs.to_string())
    print(f"--- {var} vs churn: % dentro de cada categoría ---")
    print(contingencia_rel.to_string())

# Resumen por contrato y por modalidad de pago.
resumen_contrato = (df.groupby("contract", observed=True)["churn_flag"]
                      .agg(["count", "sum", "mean"]))
resumen_contrato.columns = ["clientes", "bajas", "tasa_churn"]
resumen_contrato["tasa_churn"] *= 100

resumen_pago = (df.groupby("payment_method", observed=True)["churn_flag"]
                  .agg(["count", "sum", "mean"])
                  .sort_values("mean", ascending=False))
resumen_pago.columns = ["clientes", "bajas", "tasa_churn"]
resumen_pago["tasa_churn"] *= 100

print("\nResumen por tipo de contrato:")
print(resumen_contrato.round(2).to_string())
print("\nResumen por modalidad de pago (ordenado por tasa de churn):")
print(resumen_pago.round(2).to_string())

**Hallazgos del análisis categórico.**

- **Tasa global de churn: 26,54 %** (1.869 bajas sobre 7.043 clientes): aproximadamente **1 de cada 4 clientes** abandona el servicio.
- **Contrato:** la tasa es **42,71 % en mes a mes**, **11,27 % en contratos de un año** y **2,83 % en los de dos años**. El contrato sin permanencia multiplica el riesgo y confirma **H1**.
- **Medio de pago:** **cheque electrónico 45,29 %**, muy por encima de tarjeta automática (15,24 %), transferencia automática (16,71 %) y cheque por correo (19,11 %). El pago manual se asocia con mayor fuga y confirma **H4**.
- **Sociodemográficas:** los adultos mayores presentan 41,68 % de churn frente a 23,61 %; los clientes sin pareja, 32,96 % frente a 19,66 %; y los clientes sin dependientes, 31,28 % frente a 15,45 %. La **H6** se cumple en las tres variables, con el efecto más marcado en `senior_citizen`.
- **Género:** la diferencia es mínima (26,92 % vs. 26,16 %), por lo que **no es un segmento de riesgo relevante**.

## B.4 Visualización de datos

Construimos el reporte gráfico con **Matplotlib** y **Seaborn**. Cada figura se guarda en `reports/figures/` (para reutilizarla en el informe y la exposición) y se muestra en el notebook. Debajo de cada una se incluye su interpretación orientada al negocio.

In [ ]:
# --- B.4 Visualizaciones ---
os.makedirs("reports/figures", exist_ok=True)

# Segmentos de antigüedad para enriquecer la interpretación del histograma.
nuevos = df["tenure"] <= 6
maduros = df["tenure"] > 24
pct_nuevos = nuevos.mean() * 100
churn_nuevos = df.loc[nuevos, "churn_flag"].mean() * 100
churn_maduros = df.loc[maduros, "churn_flag"].mean() * 100

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(data=df, x="tenure", bins=30, kde=True, color="steelblue", ax=ejes[0])
ejes[0].axvline(df["tenure"].median(), color="black", linestyle="--", linewidth=1.2,
                label=f"Mediana: {df['tenure'].median():.0f} meses")
ejes[0].legend()
ejes[0].set_title("Antigüedad del cliente (tenure)")
ejes[0].set_xlabel("Meses como cliente")
ejes[0].set_ylabel("Cantidad de clientes")

sns.histplot(data=df, x="monthly_charges", bins=30, kde=True, color="darkorange", ax=ejes[1])
ejes[1].axvline(df["monthly_charges"].median(), color="black", linestyle="--", linewidth=1.2,
                label=f"Mediana: {df['monthly_charges'].median():,.2f} USD")
ejes[1].legend()
ejes[1].set_title("Cargos mensuales (monthly_charges)")
ejes[1].set_xlabel("Dólares por mes")
ejes[1].set_ylabel("Cantidad de clientes")

fig.tight_layout()
fig.savefig("reports/figures/01_histogramas_tenure_monthly.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Clientes con 6 meses o menos: {pct_nuevos:.1f}% | churn: {churn_nuevos:.1f}% "
      f"| churn con más de 24 meses: {churn_maduros:.1f}%")

bajos = df["monthly_charges"] < 40
altos = df["monthly_charges"] >= 80
print(f"Clientes con cargo < 40 USD: {bajos.mean():.1%} | churn: {df.loc[bajos, 'churn_flag'].mean():.1%}")
print(f"Clientes con cargo >= 80 USD: {altos.mean():.1%} | churn: {df.loc[altos, 'churn_flag'].mean():.1%}")

### B.4.1 Histogramas/KDE de `tenure` y `monthly_charges`

- **Antigüedad:** la distribución es bimodal. Hay una concentración importante de clientes nuevos (21,0 % tiene 6 meses o menos) y otra de clientes de largo plazo. La tasa de churn en los primeros 6 meses es **52,9 %**, frente a **14,0 %** en clientes con más de 24 meses. **El primer semestre es la etapa crítica**: ahí se concentra el mayor riesgo y, por lo tanto, el mayor retorno esperado de las acciones de onboarding y fidelización. Esto respalda **H2**.
- **Cargo mensual:** la distribución también es bimodal: 26,1 % paga menos de 40 USD y 38,0 % paga 80 USD o más. La tasa de churn es **11,6 %** en el segmento de menor cargo y **34,0 %** en el de mayor cargo. El segmento de alto valor aporta más ingreso por cliente, pero también es más sensible al precio y requiere una propuesta de valor clara (soporte, servicios adicionales, reconocimiento). Esto respalda **H3**.

In [ ]:
# --- B.4.2 Matriz de correlación de Pearson ---
fig, eje = plt.subplots(figsize=(7, 5.5))

sns.heatmap(
    matriz_corr,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Coeficiente de correlación de Pearson"},
    xticklabels=["Antigüedad\n(tenure)", "Cargo mensual\n(monthly_charges)", "Cargo total\n(total_charges)"],
    yticklabels=["Antigüedad\n(tenure)", "Cargo mensual\n(monthly_charges)", "Cargo total\n(total_charges)"],
    ax=eje,
)
eje.set_title("Matriz de correlación de Pearson")
fig.tight_layout()
fig.savefig("reports/figures/02_matriz_correlacion.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

### B.4.2 Matriz de correlación

El mapa de calor confirma visualmente lo calculado en B.2: la celda más intensa es `tenure`–`total_charges` (r = **0,826**), seguida de `monthly_charges`–`total_charges` (**0,651**); `tenure`–`monthly_charges` es débil (**0,248**). La lectura de negocio está desarrollada en B.2: **la antigüedad es el principal motor del valor acumulado**, por lo que retener clientes desde el inicio es la palanca más directa para aumentar el valor de vida.

In [ ]:
# --- B.4.3 Tasa de churn según tipo de contrato ---
orden_contrato = ["Month-to-month", "One year", "Two year"]
tasa_contrato = (df.groupby("contract", observed=True)["churn_flag"].mean()
                   .reindex(orden_contrato) * 100)

fig, eje = plt.subplots(figsize=(7, 4.5))
barras = eje.bar(tasa_contrato.index, tasa_contrato.values,
                 color=["#c44e52", "#dd8452", "#55a868"])
eje.bar_label(barras, fmt="%.1f%%", padding=3)
eje.set_title("Tasa de cancelación según tipo de contrato")
eje.set_xlabel("Tipo de contrato")
eje.set_ylabel("Tasa de cancelación (%)")
eje.set_ylim(0, tasa_contrato.max() * 1.2)
fig.tight_layout()
fig.savefig("reports/figures/03_churn_contrato.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

### B.4.3 Churn según tipo de contrato

La tasa cae de **42,7 %** en mes a mes a **11,3 %** en contratos de un año y **2,8 %** en los de dos años. La modalidad sin permanencia multiplica el riesgo de fuga: **migrar clientes a contratos anuales con incentivos** (descuentos, servicios adicionales o renovación premiada) es una de las acciones de retención con mayor impacto. Confirma **H1**.

In [ ]:
# --- B.4.4 Tasa de churn según modalidad de pago ---
tasa_pago = (df.groupby("payment_method", observed=True)["churn_flag"].mean()
               .sort_values(ascending=False) * 100)

fig, eje = plt.subplots(figsize=(9, 4.5))
barras = eje.barh(tasa_pago.index[::-1], tasa_pago.values[::-1], color="darkorange")
eje.bar_label(barras, fmt="%.1f%%", padding=3)
eje.set_title("Tasa de cancelación según modalidad de pago")
eje.set_xlabel("Tasa de cancelación (%)")
eje.set_ylabel("Modalidad de pago")
eje.set_xlim(0, tasa_pago.max() * 1.15)
fig.tight_layout()
fig.savefig("reports/figures/04_churn_modalidad_pago.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

pago_automatico = df["payment_method"].isin(["Bank transfer (automatic)", "Credit card (automatic)"])
print(f"Tasa de churn en pagos automáticos: {df.loc[pago_automatico, 'churn_flag'].mean():.1%}")

### B.4.4 Churn según modalidad de pago

El **cheque electrónico** presenta la mayor tasa de cancelación (**45,3 %**), muy por encima del promedio de los pagos automáticos (**16,0 %**). La modalidad de pago actúa como señal de compromiso: la domiciliación reduce la fricción y se asocia con mayor permanencia. **Promover el débito automático con beneficios concretos** puede disminuir la fuga de forma medible y confirma **H4**.

## B.5 Contraste de hipótesis

La siguiente tabla resume, para cada hipótesis de A.1.3, la evidencia numérica obtenida en el análisis y su veredicto. El código siguiente reproduce esos valores.

In [ ]:
# --- B.5 Contraste de hipótesis ---
# H5: con internet pero sin soporte técnico ni seguridad online.
f3 = ((df["internet_service"] != "No")
      & (df["tech_support"] == "No")
      & (df["online_security"] == "No"))

# H6: adulto mayor, sin pareja y sin dependientes.
h6 = ((df["senior_citizen"] == "Yes")
      & (df["partner"] == "No")
      & (df["dependents"] == "No"))

contraste = pd.DataFrame({
    "segmento": [
        "Global",
        "Contrato mes a mes",
        "Contrato de 2 años",
        "Churn con tenure <= 6",
        "Churn con tenure > 24",
        "Cargo mensual >= 80 USD",
        "Cheque electrónico",
        "Con internet sin soporte ni seguridad (H5)",
        "Adulto mayor sin pareja ni dependientes (H6)",
    ],
    "clientes": [
        len(df),
        int((df["contract"] == "Month-to-month").sum()),
        int((df["contract"] == "Two year").sum()),
        int((df["tenure"] <= 6).sum()),
        int((df["tenure"] > 24).sum()),
        int((df["monthly_charges"] >= 80).sum()),
        int((df["payment_method"] == "Electronic check").sum()),
        int(f3.sum()),
        int(h6.sum()),
    ],
    "tasa_churn_%": [
        df["churn_flag"].mean() * 100,
        df.loc[df["contract"] == "Month-to-month", "churn_flag"].mean() * 100,
        df.loc[df["contract"] == "Two year", "churn_flag"].mean() * 100,
        df.loc[df["tenure"] <= 6, "churn_flag"].mean() * 100,
        df.loc[df["tenure"] > 24, "churn_flag"].mean() * 100,
        df.loc[df["monthly_charges"] >= 80, "churn_flag"].mean() * 100,
        df.loc[df["payment_method"] == "Electronic check", "churn_flag"].mean() * 100,
        df.loc[f3, "churn_flag"].mean() * 100,
        df.loc[h6, "churn_flag"].mean() * 100,
    ],
}).round(2)

print(contraste.to_string(index=False))

| # | Hipótesis | Evidencia | Veredicto |
|---|---|---|---|
| **H1** | Contrato mes a mes cancela más | 42,7 % vs. 11,3 % (1 año) y 2,8 % (2 años) | **Se confirma** |
| **H2** | Más churn en los primeros meses | 52,9 % con ≤ 6 meses vs. 14,0 % con > 24 meses | **Se confirma** |
| **H3** | Cargos mensuales altos cancelan más | 34,0 % con ≥ 80 USD vs. 11,6 % con < 40 USD | **Se confirma** |
| **H4** | Cheque electrónico concentra el churn | 45,3 % vs. 16,0 % en pagos automáticos | **Se confirma** |
| **H5** | Sin soporte ni seguridad cancela más | 49,0 % en el segmento sin ambos servicios vs. 26,5 % global | **Se confirma** |
| **H6** | Adultos mayores sin pareja ni dependientes | 49,2 % en el segmento vs. 26,5 % global | **Se confirma** |

**Conclusiones para la toma de decisiones**

1. **Priorizar la retención temprana:** el churn se concentra en los primeros 6 meses (52,9 %). Un programa de onboarding con seguimiento en los meses 1–3 tiene el mayor retorno esperado.
2. **Incentivar contratos largos:** migrar de mes a mes a contrato anual o bianual reduce la tasa de 42,7 % a 11,3 %–2,8 %. Diseñar descuentos o servicios adicionales por renovación.
3. **Fomentar el pago automático:** el cheque electrónico (45,3 %) duplica el riesgo frente a la domiciliación (16,0 %). Ofrecer beneficios por débito automático.
4. **Atender a los segmentos vulnerables:** adultos mayores sin pareja ni dependientes (49,2 %) y clientes de alto cargo mensual (34,0 %) requieren propuestas de valor específicas (soporte, seguridad, precios claros).
5. **Proteger el valor de vida:** la fuerte correlación `tenure`–`total_charges` (r = 0,826) muestra que cada baja temprana no solo pierde un abono, sino años de facturación futura.

Estos resultados son descriptivos y se basan en el dataset histórico analizado; no prueban causalidad. Las acciones propuestas deberían validarse con experimentos controlados o seguimiento en el tiempo.

---
# Parte C — Reproducibilidad y buenas prácticas

Este notebook se diseñó para que **cualquier persona obtenga exactamente los mismos resultados** al ejecutarlo de principio a fin en Google Colab:

| Práctica | Cómo se aplica en este proyecto |
|---|---|
| **Entorno reproducible** | Corre en Colab sin instalar nada: Pandas, NumPy, Matplotlib y Seaborn vienen preinstalados. Las versiones probadas figuran en `requirements.txt` y se imprimen en la celda siguiente. |
| **Datos accesibles** | El CSV se descarga desde una URL pública fija; si existe una copia local en `data/raw/`, se usa esa. No hace falta subir archivos a mano ni tener credenciales de Kaggle. |
| **Datos crudos inmutables** | `df_raw` nunca se modifica; todas las transformaciones se aplican sobre la copia `df`. |
| **Determinismo** | Semilla fija (`SEMILLA = 42`) para cualquier operación aleatoria. |
| **Ejecución secuencial** | Cada celda depende solo de las anteriores. Se valida con *Entorno de ejecución → Reiniciar y ejecutar todas* antes de cada entrega. |
| **Rutas relativas** | No hay rutas absolutas de ninguna computadora (`C:\Users\...`): el proyecto funciona en cualquier máquina. |
| **Decisiones documentadas** | Cada paso de limpieza o transformación tiene una celda de texto que justifica la decisión tomada. |
| **Control de versiones** | Repositorio Git en GitHub con commits descriptivos. |

In [ ]:
import sys, platform, matplotlib

print("Entorno de ejecución")
print(f"  Python      {sys.version.split()[0]}  ({platform.system()})")
print(f"  pandas      {pd.__version__}")
print(f"  numpy       {np.__version__}")
print(f"  matplotlib  {matplotlib.__version__}")
print(f"  seaborn     {sns.__version__}")
print(f"  En Colab:   {'google.colab' in sys.modules}")

In [ ]:
# Controles automáticos: si alguno falla, el notebook se detiene y avisa
assert df_raw.shape == (7043, 21), "El dataset original no tiene la forma esperada"
assert df.isna().sum().sum() == 0, "Quedaron valores nulos después de la limpieza"
assert pd.api.types.is_float_dtype(df["total_charges"]), "total_charges no es numérica"
assert df["churn_flag"].isin([0, 1]).all(), "churn_flag tiene valores no binarios"
print("Todos los controles de calidad pasaron correctamente.")